In [15]:
import finnhub
import json
import os
import time
from typing import Dict, Any
from dotenv import load_dotenv

load_dotenv()
FINNHUB_API_KEY = os.getenv('FINNHUB_API_KEY')
CACHE_FILE = "aapl_quote_cache.json"
CACHE_TTL_SECONDS = 3 * 60 * 60  # 3 hours


In [16]:
def get_apple_stock_price(api_key: str) -> Dict[str, Any]:
    # Check if cache exists and is still valid
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, "r") as f:
                cache = json.load(f)
            cached_time = cache.get("timestamp", 0)
            if time.time() - cached_time < CACHE_TTL_SECONDS:
                # Return cached data (remove timestamp before returning)
                cached_data = cache.get("data", {})
                if cached_data:
                    return cached_data
        except (json.JSONDecodeError, KeyError, OSError):
            # If cache is corrupted, ignore and fetch fresh
            pass

    # Fetch fresh data from FinHub
    finnhub_client = finnhub.Client(api_key=FINNHUB_API_KEY)
    try:
        quote = finnhub_client.quote("AAPL")
        print(f"Fetched quote: ", quote)
    except Exception as e:
        # Re-raise with a clear message
        raise RuntimeError(f"Failed to fetch quote from FinHub: {e}")

    # Save to cache with timestamp
    cache_data = {
        "timestamp": time.time(),
        "data": quote
    }
    try:
        with open(CACHE_FILE, "w") as f:
            json.dump(cache_data, f)
    except OSError as e:
        # Non-critical; we can continue without caching
        print(f"Warning: could not write cache file: {e}")

    return quote

In [ ]:

# quote = get_apple_stock_price(api_key)
# print(quote)

Fetched quote:  {'c': 281.74, 'd': -2.04, 'dp': -0.7189, 'h': 288.3697, 'l': 279.85, 'o': 286.73, 'pc': 283.78, 't': 1782763200}
{'c': 281.74, 'd': -2.04, 'dp': -0.7189, 'h': 288.3697, 'l': 279.85, 'o': 286.73, 'pc': 283.78, 't': 1782763200}
